# 04 — Seleção do vencedor e decisão de promoção

**Frente:** Dev 3 · **Critérios:** 5 (vencedor escolhido pela validação) e 8 (decisão automática e rastreada)

Roda a cada ciclo, logo depois do `03_treino_competicao.ipynb`:

1. **Vencedor do ciclo**: entre A e B, o de maior **F1 de validação** (empate → modelo A, o mais simples).
2. **Modelo vigente**: carregado do MLflow pelo `run_id` de `models/modelo_vigente.json` e reavaliado na **mesma** validação do ciclo atual.
3. **Regra de promoção** (fixada no `config.yaml` antes do ciclo 1): o vencedor substitui o vigente **só se** `F1_val ≥ F1_vigente + 0,01` **e** `F1_val ≥ 0,60`. Sem vigente (ciclo 1), basta `F1_val ≥ 0,60`.
4. **Teste**: só depois da decisão, e só do modelo que ficou vigente, para o número do relatório.
5. **Manifesto e histórico**: `models/modelo_vigente.json` e `models/historico_decisoes.csv`, escritos **no último passo**. Se qualquer célula falhar antes, nada muda no disco e o workflow não commita (sem promoção indevida).

Nenhum modelo é apagado: o manifesto só troca o ponteiro (`run_id`); os runs anteriores continuam no MLflow.

In [ ]:
import os, sys, json
from pathlib import Path

# funciona tanto rodando de notebooks/ (nbconvert) quanto da raiz
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
os.environ.setdefault("MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR", "false")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.ingestao import carregar_config
from src.preprocessamento import carregar_particoes, carregar_schema
from src.modelagem import carregar_competicao, METRICAS
from src.promocao import (carregar_regra_promocao, escolher_vencedor, carregar_manifesto,
                          executar_promocao, registrar_decisao_mlflow, gravar_decisao,
                          carregar_historico, caminho_manifesto, PROMOVIDO)

cfg = carregar_config()

# cycle_id: o do workflow (CYCLE_ID) ou, localmente, o da competição mais recente do notebook 03
competicao = carregar_competicao(cfg=cfg)
CYCLE_ID, DATA_VERSION = competicao["cycle_id"], competicao["data_version"]
assert DATA_VERSION == carregar_schema(cfg)["data_version"], "competição de outro data_version: rode o 03"
print("cycle_id:    ", CYCLE_ID)
print("data_version:", DATA_VERSION)
print("features:    ", competicao["features"])
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)

## 1. Regra (fixada antes de ver resultados)

Lida de `config.yaml` → `promocao` por `carregar_regra_promocao`. O teste `tests/test_promocao.py::test_regra_fixada_no_config` trava esses valores: mudar a regra depois de ver resultados quebra a suíte.

- **margem de 0,01**: evita trocar de modelo por ruído (diferenças de F1 dessa ordem aparecem só por mudança de amostra na validação); troca de modelo tem custo (revalidar, comunicar), então só vale com ganho mensurável.
- **piso de 0,60**: nenhum modelo com F1 abaixo disso vai para uso, nem no ciclo 1.

In [ ]:
regra = carregar_regra_promocao(cfg)
pd.Series(regra, name="valor").to_frame()

## 2. Vencedor do ciclo (só validação)

In [ ]:
cand = competicao["candidatos"]
tabela = pd.DataFrame({n: {"algoritmo": c["algoritmo"], **{m: round(c["metricas_validacao"][m], 4) for m in METRICAS}}
                       for n, c in cand.items()}).T
VENCEDOR = escolher_vencedor(cand, regra)
tabela["vencedor"] = tabela.index == VENCEDOR
print(f"vencedor do ciclo: modelo {VENCEDOR} ({cand[VENCEDOR]['algoritmo']}), "
      f"F1_val = {cand[VENCEDOR]['metricas_validacao']['f1']:.4f}")
tabela

## 3. Modelo vigente e decisão

`executar_promocao` carrega o manifesto, recarrega o vigente do MLflow, mede-o na validação deste ciclo, aplica a regra e calcula o novo manifesto **sem gravar nada ainda**.

In [ ]:
anterior = carregar_manifesto(cfg)
if anterior:
    print(f"vigente atual: modelo {anterior['modelo']} ({anterior['algoritmo']}), run {anterior['run_id']}")
    print(f"  treinado no ciclo {anterior['treinado_no_ciclo']} ({anterior['treinado_com_data_version']}), "
          f"features: {anterior['features']}")
else:
    print("não há modelo vigente (primeiro ciclo)")

_, validacao, teste = carregar_particoes(cfg)   # teste: só usado após a decisão, no vigente resultante
resultado = executar_promocao(competicao, validacao, teste, cfg, manifesto=anterior)
man = resultado["manifesto"]
print()
print("DECISÃO:", man["decisao"].upper())
print("motivo: ", man["motivo"])
print(f"F1 necessário para promover: {man['f1_necessario']:.4f}")

### Verificações de qualidade do vencedor

Além do limiar de F1, o vencedor só é promovido se passar em **todas** as verificações de `config.yaml` → `promocao.verificacoes_qualidade`:

- **precision e recall mínimos**: um F1 alto não pode esconder um modelo que acusa fraude demais ou deixa passar a maioria;
- **taxa prevista de fraude** dentro de uma faixa: rejeita modelo degenerado, que prevê uma classe só;
- **métricas finitas**;
- **recarga do MLflow**: o candidato é recarregado pelo `run_id` e precisa reproduzir o F1 de validação. Isso prova que o modelo serializado é o mesmo que foi avaliado (é ele que o notebook 05 vai usar).

Se alguma reprovar, a decisão é `mantido`, com as falhas no motivo.

In [ ]:
q = man["qualidade"]
print("verificações de qualidade:", "APROVADO" if q["aprovada"] else "REPROVADO")
pd.DataFrame(q["checagens"])[["nome", "ok", "valor", "limite"]]

In [ ]:
linhas = {f"{n} · {c['algoritmo']}": c["metricas_validacao"] for n, c in cand.items()}
if man["vigente_anterior"]:
    va = man["vigente_anterior"]
    linhas[f"vigente · {va['algoritmo']} ({va['treinado_no_ciclo']})"] = va["metricas_validacao"]
comp_val = pd.DataFrame(linhas).T[["precision", "recall", "f1", "roc_auc"]].astype(float).round(4)
display(comp_val)

fig_dec, ax = plt.subplots(figsize=(8.5, 4))
x = np.arange(3); w = 0.8 / len(comp_val)
cores = ["tab:blue", "tab:orange", "tab:gray"]
for i, (rot, row) in enumerate(comp_val.iterrows()):
    b = ax.bar(x + (i - (len(comp_val) - 1) / 2) * w, row[["precision", "recall", "f1"]], w, label=rot, color=cores[i])
    ax.bar_label(b, fmt="%.3f", fontsize=7)
ax.axhline(man["f1_necessario"], ls="--", color="k", lw=1, label=f"F1 necessário ({man['f1_necessario']:.3f})")
ax.set_xticks(x, ["precision", "recall", "f1"]); ax.set_ylim(0, 1.1); ax.legend(fontsize=8, loc="lower right")
ax.set_title(f"Validação do ciclo {CYCLE_ID}: candidatos vs vigente — {man['decisao']}")
plt.tight_layout(); plt.show()

## 4. Teste (só para o relatório)

Medido **depois** da decisão e **apenas** no modelo que ficou vigente. Esse número não influencia nada no pipeline.

In [ ]:
if man["run_id"]:
    print(f"vigente após a decisão: modelo {man['modelo']} ({man['algoritmo']}), run {man['run_id']}")
    display(pd.DataFrame({"validação": man["metricas_validacao"], "teste": man["metricas_teste"]}).T
            [["precision", "recall", "f1", "roc_auc", "tp", "fp", "fn", "tn"]].round(4))
else:
    print("nenhum modelo vigente: o vencedor não atingiu o F1 mínimo")

## 5. Registro da decisão e manifesto (último passo)

1. Run MLflow `promocao_<cycle_id>` (tags `cycle_id`, `data_version`, `decisao`; métricas `promovido`, `f1_val_*`, `f1_teste_vigente`; artefato `decisao.json`). Se promoveu, o modelo é registrado no Model Registry (`fraude_ethereum`) e o alias `vigente` passa a apontar para ele.
2. `artifacts/<cycle_id>/decisao.json`, uma linha nova em `models/historico_decisoes.csv` e, por último, `models/modelo_vigente.json`.

In [ ]:
decisao_run = registrar_decisao_mlflow(resultado, cfg, figuras={"figuras/decisao_validacao.png": fig_dec})
caminhos = gravar_decisao(resultado, cfg)
print("run da decisão:", decisao_run)
for k, p in caminhos.items():
    print(f"{k:10s}", p.relative_to(RAIZ).as_posix())

manifesto = json.loads(caminho_manifesto(cfg).read_text(encoding="utf-8"))
print(json.dumps({k: manifesto[k] for k in ("cycle_id", "data_version", "decisao", "motivo", "run_id", "modelo",
                                             "algoritmo", "features", "f1_validacao", "f1_teste",
                                             "treinado_no_ciclo", "registry")},
                 indent=2, ensure_ascii=False))

In [ ]:
historico = carregar_historico(cfg)
historico[["cycle_id", "data_version", "f1_val_A", "f1_val_B", "vencedor_algoritmo", "f1_val_vigente_anterior",
           "decisao", "vigente_algoritmo", "vigente_treinado_no_ciclo", "f1_val_vigente", "f1_teste_vigente"]]

### Entregas para as outras frentes
- **Dev 4 (notebook 05):** `models/modelo_vigente.json` → `mlflow.sklearn.load_model(manifesto["model_uri"])` (ou `src.modelagem.carregar_modelo(manifesto["run_id"])`) e `predict_proba` direto nas partições: o pipeline já faz o pré-processamento e a seleção das features.
- **Dev 4 (workflow):** persistir `models/`, `artifacts/<cycle_id>/`, `mlflow.db` e `mlartifacts/`. O manifesto é o último arquivo escrito; se o notebook falhar antes, ele fica como estava.
- **Dev 5 (painel):** `models/modelo_vigente.json` (tela Visão geral), `models/historico_decisoes.csv` (tela Ciclos), `artifacts/<cycle_id>/decisao.json` (métricas do vigente na tela Competição).
- **Relatório:** seção "Decisão de promoção" (`docs/competicao_promocao.md`).